# Battle 采样节点（rollout / eval）— Kaggle / Colab（薄 cell 版）

这台云机**只采样本与跑评估**，不训练：它起 `tools/agent/sampler-agent.ts`，把它的 HTTP 端口暴露到
公网（Colab 优先 tailscale；其余平台、或 tailscale 引导失败/入站不可达 ⇒ cloudflared 隧道），再
**自动**把 `{id,url,authKey}` 注册进训练机的 `nn-training/rl-config.json`
（`POST /admin/nodes/register`，hub 侧 upsert `nodes[]`）。

| 步骤 | 谁做 | 怎么做 |
|---|---|---|
| 下载代码库 | 本 cell | `git clone --depth 1`（**全仓**；理由见 cell 内注释） |
| 装 bun / cloudflared | `remote/rollout_node.py` | 公网下载一律包在 `platform_net_env()` 里（代理注入后公网不通） |
| 网络通道 | `remote/rollout_node.py` | tailscale 优先 → 注册被 hub ping 门拒（422）⇒ 回落 cloudflared 并重注册 |
| 并发度 | `remote/rollout_node.py` | `platform_utils.cpu_worker_slots()`（rollout/eval 的**唯一口径**） |
| 保活 / 日志 | 本 cell | 心跳打印 + 同时落文件（Colab 输出会连着丢） |
| 自动注册 | `remote/rollout_node.py` | `HUB_IP`/`HUB_TOKEN` secret 齐 ⇒ 自动；缺 ⇒ 打印 ★URL + ★authKey 走手工路径 |

**逻辑住在仓库**：`nn-training/remote/rollout_node.py` —— cell 只留 CFG / 凭据 / 保活 / clone / 调用。
改逻辑请改那个文件，**别在这里内联**（`tests/remote/test_tailscale_boot_bun.py` 会红）。

**用法**：填 `CFG.repo`（+ 私有仓的 `github_token`）→ 配好 secret `HUB_IP` / `HUB_TOKEN`
（tailscale 通道另需 `TS_AUTHKEY`）→ Run All → 等日志出现 `★ 已自动注册`。

**手工兜底**（没配 hub 时照旧可用）：把日志里的 `★ URL` 与 `★ authKey` 写进本机
`nn-training/rl-config.json`：

```json
{ "nodes": [ { "id": "rollout-colab", "url": "<URL>", "authKey": "<authKey>" } ] }
```

（`concurrency` 不填 = 按节点核数；节点门要 `ping` 通且 `evalSupport` 为真 —— 见
`trainer/batch_plan.py:57-61` 与 `trainer/eval_dispatch.py:114`。）

**停止**：■ 中断本 cell → 自动 unregister（`enabled=false` + `unregistered_at`，条目**不删**：下个
会话注册时恢复；要永久停用请在控制台删掉那条）。会话按 `max_session_hours` 到点也会干净收线。


In [ ]:
# @title Battle 采样节点 —— 填 CFG.repo（+ 私有仓 token）后直接 Run
import os, subprocess, sys, threading, time
from pathlib import Path

CFG = {
    # ── 代码库 ──────────────────────────────────────────────────────────
    # 采样节点要的是 **tools/（TS 导出器）+ src/（wasm 内核）+ nn-training/**：
    # hub 的 `GET /code` 只有 nn-training 下的 .py/.jsonc，而 TS 运行时在 ts_code.zip 里
    # 且只随具体 job 下发 —— 采样节点没有 job，所以直接 clone 全仓是唯一省事的路
    # （云机有外网）。
    "repo": "https://github.com/<owner>/battle2.git",
    "ref": "main",
    "github_token": "",        # 私有仓 PAT；留空则读环境变量 GITHUB_TOKEN
    "code_dir": "/kaggle/working/battle2",   # Colab 建议 /content/battle2
    # ── 网络通道与自动注册 ───────────────────────────────────────────────
    "link": "auto",            # auto | tailscale | cloudflared（auto：Colab 优先 tailscale）
    "node_id": "",             # 留空 = rollout-<env>（colab/kaggle）
    "hub_ip": "",              # 推荐用 secret HUB_IP；"100.x.y.z" 或整条 "http://host:8900"
    "hub_port": 8900,
    "hub_url": "",             # 整条 URL 时填这里（优先于 hub_ip + hub_port）
    "hub_token": "",           # 推荐用 secret HUB_TOKEN（= 训练机 HUB_TOKEN）
    "ts_authkey": "",          # 推荐用 secret TS_AUTHKEY；cloudflared 通道可留空
    "ts_ephemeral": True,      # tailnet 退出即注销（别留僵尸节点名）
    "ts_proxy_env": True,      # userspace 模式注入代理环境（出站可用、入站不通）
    "ts_engine": "",           # 空 = auto（用户态优先）；"kernel" 等可强压
    "ts_no_proxy_extra": "",   # 追加 NO_PROXY（逗号分隔）
    "ts_tarball": "",          # 预置 tarball 路径/URL（内网/离线环境）
    "ts_prefix": "",           # 预置 tailscale 安装前缀
    # ── agent ───────────────────────────────────────────────────────────
    "agent_port": 8443,
    "workers": 0,              # 0 = 自动（platform_utils.cpu_worker_slots；>0 完全照填的数）
    "cache_mb": 2048,          # 结果缓存上限（同键重入直接回放）
    "agent_extra": [],         # 例 ["--no-node"] = 采样子进程强制 bun（A/B 与回滚用）
    "bun_install_deps": False, # agent 零第三方依赖；勾 True 才跑 bun install
    # ── 会话与保活 ───────────────────────────────────────────────────────
    "max_session_hours": 9,    # 云机单次会话上限；到点干净收线（含 unregister）
    "restart_limit": 5,        # agent 崩溃自动重启上限（超过就停 cell，别无限重启）
    "keepalive_every_sec": 120,
    "cloudflared_path": "",    # 留空 = 自动找/自动装（实现住 remote/rollout_node.py）
    "log_file": "",            # 留空 = /content → /kaggle/working → /tmp 自动挑
}

# 日志同时落文件：会话被无声回收时 cell 输出会连着丢（plan §2.5）
def _pick_log_path():
    for _d in ("/content", "/kaggle/working"):
        if Path(_d).is_dir():
            return Path(_d) / "rollout-node.log"
    return Path("/tmp/rollout-node.log")


_LOG_PATH = Path(CFG.get("log_file") or "") if CFG.get("log_file") else _pick_log_path()


def _log(msg):
    line = f"[{time.strftime('%H:%M:%S')}] [rollout-node] {msg}"
    print(line, flush=True)
    try:
        with _LOG_PATH.open("a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except OSError:
        pass


def _secret(key, cfg_val=""):
    """环境变量 → Colab/Kaggle Secrets → CFG 手填（值永不进日志，只记**来源**）。

    ★ 调用发生在 `run()` 的最前面、任何代理注入（tailscale userspace 会改写 HTTP_PROXY）
    之前 —— 顺序是契约（2026-09-17 事故同族），别把凭据读取挪到 clone/引导之后。
    """

    def _env():
        return os.environ.get(key, "")

    def _colab():
        from google.colab import userdata
        return userdata.get(key) or ""

    def _kaggle():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key) or ""

    for _name, _get in (("env", _env), ("colab", _colab), ("kaggle", _kaggle)):
        try:
            _v = _get()
        except Exception as _e:
            _log(f"凭据 {key} ← {_name} 取用失败（{type(_e).__name__}）")
            _v = ""
        if _v:
            _log(f"凭据 {key} ← {_name}")
            return str(_v).strip()
    if cfg_val:
        _log(f"凭据 {key} ← CFG 手填")
    return str(cfg_val or "").strip()


# ── 保活：云机一段时间无输出会被回收 —— 打印本身就是心跳 ────────────────
# （与 battle.cloudflared.ipynb / battle.offline.ipynb 同一招；Colab 那个分支要点连接
#  按钮，Kaggle 只要别静默。）
_stop = threading.Event()


def _keepalive():
    n = 0
    every = max(30, int(CFG["keepalive_every_sec"]))
    while not _stop.is_set():
        _log(f"[keepalive] alive ({n * every // 60} min)")
        n += 1
        _stop.wait(every)


threading.Thread(target=_keepalive, daemon=True, name="keepalive").start()
_log("Keepalive 已启动")


# ── 代码库（GITHUB_TOKEN 只在这里用；clone 走平台网络，代理还没注入）────────
def _sh(cmd, *, cwd=None, timeout=None):
    try:
        return subprocess.run(cmd, cwd=cwd, capture_output=True, text=True,
                            encoding='utf-8', errors='replace', timeout=timeout)
    except (OSError, subprocess.SubprocessError) as e:

        class _R:
            returncode = 127
            stdout = ""
            stderr = str(e)

        return _R()


code_dir = Path(CFG["code_dir"])
repo = str(CFG["repo"] or "").strip()
ref = str(CFG["ref"] or "main").strip() or "main"
if (not repo) or "<owner>" in repo:
    raise SystemExit("[rollout-node] FATAL: CFG.repo 必填（形如 https://github.com/<owner>/battle2.git）")
_tok = str(CFG["github_token"] or "").strip() or os.environ.get("GITHUB_TOKEN", "")
_url = repo
if _tok and repo.startswith("https://"):
    _url = repo.replace("https://", f"https://{_tok}@")   # 私有仓；只在内存里拼，不落日志
code_dir.parent.mkdir(parents=True, exist_ok=True)
if (code_dir / ".git").is_dir():
    _sh(["git", "-C", str(code_dir), "fetch", "--depth", "1", "origin", ref], timeout=600)
    _sh(["git", "-C", str(code_dir), "checkout", "--force", ref], timeout=300)
    _sh(["git", "-C", str(code_dir), "reset", "--hard", f"origin/{ref}"], timeout=300)
else:
    r = _sh(["git", "clone", "--depth", "1", "--branch", ref, _url, str(code_dir)], timeout=900)
    if r.returncode != 0:
        _log("git clone 失败（私有仓请填 github_token / secret GITHUB_TOKEN）：")
        _log(r.stderr[-600:])
        raise SystemExit(2)
_head = _sh(["git", "-C", str(code_dir), "rev-parse", "--short", "HEAD"]).stdout.strip()
_log(f"代码库就绪: {code_dir}  commit={_head}  ref={ref}")


# ── 调用仓库里的运行时：bun / cloudflared / 通道 / agent / 注册 / 监护全在那边 ──
sys.path.insert(0, str(code_dir / "nn-training"))
from remote import rollout_node  # noqa: E402  （必须在 clone 之后）

try:
    _rc = rollout_node.run(CFG, _log, _secret)
finally:
    _stop.set()   # 停心跳（进程退出时 finally 里的 unregister/清理由 run 负责）
_log(f"退出码 {_rc}（0 = 正常收线）")
